# **Modelo clasificatorio de calidad de manzanas**

In [30]:
# Instalación de librerías básicas
!pip install torch torchvision pandas numpy matplotlib scikit-learn seaborn

import torch
import torch.nn as nn
import torch.optim as optim
from  torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# **Definición de las rutas del dataset y etiquetas de calidad**

In [ ]:
# Definir path
csv_path_apple = "Dataset_Apple.csv"
csv_ranks = "intervalsQuality.csv"

In [32]:
# Cargar datos
ranks_df = pd.read_csv(csv_ranks)
dataset_df = pd.read_csv(csv_path_apple)

In [33]:
# Asignación de categorias
def define_categories(dry_matter, rangos_df):

    for _, fila in rangos_df.iterrows():
        if fila['Min_DryM'] <= dry_matter <= fila['Max_DryM']:
            return fila['Category']
    return 'Desconocida'    

In [34]:
# Aplicar etiquetado
dataset_df['Category'] = dataset_df['Dry matter'].apply(
    lambda x: define_categories(x, ranks_df)
)

In [35]:
from google.colab import files

# Guardar como CSV
df = dataset_df.copy()
df.to_csv('Apple_Etiquetado.csv', index=False)

# Descargar el archivo
files.download('Apple_Etiquetado.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
def separated_features_tags(dataset_df):
    """ 
    x: Caracteristicas (Bandas espectrales)
    y: Etiquetas (Caracteristicas)
    features_columns: Nombres de las columnas features
    """
    # Manejo de errores del archivo
    if not isinstance(dataset_df, pd.DataFrame):
        raise TypeError(f"El dataset cargado debe ser un pandas dataframe")
    

    columns_delete = ['Apple', 'Dry matter', 'Category']
    columns_delete = [col for col in columns_delete if col in dataset_df.columns]
    
    # Features: todas las columnas excepto las especificadas
    X = dataset_df.drop(columns=columns_delete)
    feature_columns = X.columns.tolist()

    #Target: La columna 'Category' 
    if 'Category' in dataset_df.columns:
        Y = dataset_df['Category']
    else:
        raise KeyError("Error: No se encontro la columna 'Category' en el dataset")

    print(f"Feactures: {X.shape[1]} bandas espectrales")
    print(f"Target: {len(Y.unique())} categorias")

    return X.values, Y, feature_columns

In [37]:
def encode_tags(y):
    # Pasar string a numericas
    lb_encoder = LabelEncoder()
    y_encoder = lb_encoder.fit_transform(y)

    print(f"Etiquetas codificadas: {lb_encoder.classes_} ")
    print(f"   Mapeo: {dict(zip(lb_encoder.classes_, range(len(lb_encoder.classes_))))}")
    
    return y_encoder, lb_encoder

In [38]:
# Normalizar feactures
def normalize_feactures(x):
    scaler = StandardScaler()
    x_normalize = scaler.fit_transform(x)

    print(f"Normalizar datos - media: {x_normalize.mean():.4f}, std: {x_normalize.std():.4f}")

    return x_normalize, scaler

In [39]:
# Dividir dataset
def split_data(x, y, test_size=0.15, val_size=0.15, random_state=42):
    # Separar datos para la prueba
    x_temp, x_test, y_temp, y_test = train_test_split(
        x, y, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=y
    )

    # Separar datos para la validación
    val_relative_size = val_size / (1-test_size)
    x_train, x_val, y_train, y_val = train_test_split(
        x_temp, y_temp, 
        test_size=val_relative_size, 
        random_state=random_state, 
        stratify=y_temp
    )

    # Revisar como lo dividi
    total_samples = len(x)
    print(f"Train: {len(x_train)} muestras ({len(x_train)/total_samples*100}%)")   
    print(f"Val: {len(x_val)} muestras ({len(x_val)/total_samples*100}%)")   
    print(f"Test: {len(x_test)} muestras ({len(x_test)/total_samples*100}%)")   

    return x_train, x_val, x_test, y_train, y_val, y_test


In [40]:
def created_dataloader(x_train, x_val, x_test, y_train, y_val, y_test, batch_size=32):
    # Convertir a tensores de PyTorch
    X_train_tensor = torch.FloatTensor(x_train) #Revisar si estan o no en mayusculas el 'x' o 'y' porque deben estar en minusculas (Borrar luego)
    X_val_tensor = torch.FloatTensor(x_val)
    X_test_tensor = torch.FloatTensor(x_test)
    
    y_train_tensor = torch.LongTensor(y_train)
    y_val_tensor = torch.LongTensor(y_val)
    y_test_tensor = torch.LongTensor(y_test)
    
    # Crear datasets
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    
    # Crear DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Batch size: {batch_size}")
    print(f"   Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
    
    return train_loader, val_loader, test_loader

In [41]:
def created_pipeline_train(dataset_df, test_size=0.15, val_size=0.15, random_state=42):

    if not isinstance(dataset_df, pd.DataFrame):
        raise TypeError(f"El dataset cargado debe ser un pandas dataframe")   

    print(f"Dataset shape: {dataset_df.shape}") 

    # Paso 1: Separación de etiquetas y caracteristicas
    X, Y, feature_columns = separated_features_tags(dataset_df)
    #Paso 2: Codificar las etiquetas de calidad (category)
    y_encoded, lb_encoder = encode_tags(Y)
    #Paso 3: Normalizamos los datos
    X_norm, scaler = normalize_feactures(X)
    #Paso 4:  Dividir dataset train-val-test
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(
        X_norm, y_encoded, test_size, val_size, random_state
    )
    #Paso 5: Crear dataloaders
    train_loader, val_loader, test_loader = created_dataloader(
        X_train, X_val, X_test, y_train, y_val, y_test
    ) 

    return {
        'loaders': {
            'train': train_loader,
            'val': val_loader, 
            'test': test_loader
        },
        'preprocessors': {
            'scaler': scaler,
            'label_encoder': lb_encoder
        },
        'data_info': {
            'feature_columns': feature_columns,
            'input_size': X.shape[1],
            'num_classes': len(lb_encoder.classes_),
            'class_names': lb_encoder.classes_
        },
        'raw_splits': {
            'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
            'y_train': y_train, 'y_val': y_val, 'y_test': y_test
        }
    }

In [ ]:
# Probar como esta lo programado y si todo funciona bien hasta ahora ....
# ******************************************************************
# 1. PRIMERO carga el dataset
dataset_etiquetado = pd.read_csv("Apple_Etiquetado.csv")
print(f"Dataset cargado: {dataset_etiquetado.shape}")

# 2. Verifica las columnas que tienes
print("\nColumnas disponibles:")
print(dataset_etiquetado.columns.tolist())

# 3. Verifica las categorías
if 'Category' in dataset_etiquetado.columns:
    print(f"\nCategorías encontradas: {dataset_etiquetado['Category'].unique()}")
else:
    print("No se encuentra la columna 'Category'")

# 4. EJECUTA el pipeline con el DataFrame
print("\nEjecutando pipeline...")
pipeline = created_pipeline_train(dataset_etiquetado)

# 5. Accede a los resultados
print(f"\nPipeline completado!")
print(f"   Input size: {pipeline['data_info']['input_size']}")
print(f"   Número de clases: {pipeline['data_info']['num_classes']}")
print(f"   Clases: {pipeline['data_info']['class_names']}")

Dataset cargado: (240, 144)

Columnas disponibles:
['Apple', 'Dry matter', '430', '434', '438', '442', '446', '450', '454', '458', '462', '466', '470', '474', '478', '482', '486', '490', '494', '498', '502', '506', '510', '514', '518', '522', '526', '530', '534', '538', '542', '546', '550', '554', '558', '562', '566', '570', '574', '578', '582', '586', '590', '594', '598', '602', '606', '610', '614', '618', '622', '626', '630', '634', '638', '642', '646', '650', '654', '658', '662', '666', '670', '674', '678', '682', '686', '690', '694', '698', '702', '706', '710', '714', '718', '722', '726', '730', '734', '738', '742', '746', '750', '754', '758', '762', '766', '770', '774', '778', '782', '786', '790', '794', '798', '802', '806', '810', '814', '818', '822', '826', '830', '834', '838', '842', '846', '850', '854', '858', '862', '866', '870', '874', '878', '882', '886', '890', '894', '898', '902', '906', '910', '914', '918', '922', '926', '930', '934', '938', '942', '946', '950', '954', '

In [44]:
# Para revisar mis .csv en archivos temporales de colab
!ls -lh

total 404K
-rw-r--r-- 1 root root 400K Nov 25 04:12 Apple_Etiquetado.csv
drwxr-xr-x 1 root root 4.0K Nov 20 14:30 sample_data
